In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

uploaded = files.upload()

df = pd.read_csv("housing-2.csv")

print("Dataset loaded successfully!")
df.head()


In [ ]:
print("First 5 rows:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Values:")
print(df.duplicated().sum())

print("\nColumn Names:")
print(df.columns.tolist())



In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

# Identify target column
target = "median_house_value"

# Separate input features and target
X = df.drop(columns=[target])
y = df[target]

print("Input Features:")
print(X.columns.tolist())

print("\nTarget Column:")
print(target)

print("\nNumber of Features:", X.shape[1])

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df["median_house_value"], kde=True)

plt.title("Distribution of Median House Values")
plt.xlabel("Median House Value")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df["median_income"],
    df["median_house_value"],
    alpha=0.3
)

plt.title("Median Income vs House Value")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")

plt.show()

In [ ]:
plt.figure(figsize=(10,7))

numeric_df = df.select_dtypes(include=np.number)

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate numerical and categorical columns
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Missing values handled successfully!")
print("Categorical data encoded successfully!")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression

# Create Linear Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train the model
model.fit(X_train, y_train)

print("Linear Regression Model Trained Successfully!")

In [ ]:
# Predict using testing data
y_pred = model.predict(X_test)

# Compare actual and predicted values
result = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

print(result.head(10))

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("Model Evaluation Results")

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(y_test, y_pred, alpha=0.3)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

plt.title("Actual vs Predicted House Values")
plt.xlabel("Actual House Values")
plt.ylabel("Predicted House Values")

plt.show()

In [ ]:
# Take an existing row as a sample
new_house = X.iloc[[0]].copy()

# Change house details
new_house.loc[:, "longitude"] = -122.23
new_house.loc[:, "latitude"] = 37.88
new_house.loc[:, "housing_median_age"] = 30
new_house.loc[:, "total_rooms"] = 2000
new_house.loc[:, "total_bedrooms"] = 400
new_house.loc[:, "population"] = 800
new_house.loc[:, "households"] = 350
new_house.loc[:, "median_income"] = 5.0
new_house.loc[:, "ocean_proximity"] = "NEAR BAY"

# Predict house value
predicted_price = model.predict(new_house)

print("Predicted Median House Value:")
print(predicted_price[0])